# CellHashtag v3.0 Demo

LLM-based single-cell annotation with LATS tree search and DeepAgents.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent / 'src'))

from cellhashtag import CellHashtagAgent
import scanpy as sc

## Load Example Data

In [ ]:
adata = sc.read_h5ad('../data/example.h5ad')
print(adata)
print(f'Clusters: {adata.obs["leiden"].nunique()}')

## Run CellHashtag Agent

The agent will:
1. Perceive data (infer omics type, extract markers)
2. Iterative clustering with quality assessment
3. Annotate each cluster using DeepAgents + LATS tree search
4. Output results to annotated h5ad + CSV table

In [ ]:
agent = CellHashtagAgent(profile='fast')  # Use 'fast', 'default', or 'deep'

result = agent.run(
    input_path='../data/example.h5ad',
    output_dir='../output',
    cluster_key='leiden',
)

print(f'Status: {result["status"]}')
print(f'Clusters annotated: {result["n_clusters"]}')

## View Results

In [ ]:
import pandas as pd

table_path = Path('../output/annotation_table.csv')
if table_path.exists():
    df = pd.read_csv(table_path)
    print('Annotation Table:')
    display(df)
else:
    print('No output yet - run the agent first')

In [ ]:
adata_annotated = sc.read_h5ad('../output/annotated.h5ad')
print(adata_annotated)
print('\nAnnotation column:', 'Cell#' in adata_annotated.obs.columns)

## Configuration

CellHashtag v3.0 uses Pydantic models for configuration. Edit `src/cellhashtag/config/config.yaml` or pass overrides:

In [ ]:
# Example: custom configuration
agent_custom = CellHashtagAgent(
    profile='default',
    llm__model='qwen-plus',  # Override LLM model
    annotation__max_anno_iter=3,  # Max annotation iterations
)

print(agent_custom.config.llm.model)
print(agent_custom.config.annotation.max_anno_iter)

## Architecture

v3.0 uses a flat 4-node orchestrator:

```
perception → clustering → annotation → output
```

- **Perception**: Load h5ad, infer omics type, extract markers
- **Clustering**: Iterative Leiden with silhouette + modularity quality checks
- **Annotation**: DeepAgents + LATS MCTS tree search per cluster
- **Output**: Save annotated h5ad + CSV table

See `.research_harness_system/ARCHITECTURE.md` for full details.